### Step 1: Install Dependencies

In [21]:
# Installs PDF processing, local vector storage, and the Google Gemini API libraries.
!pip -q install pypdf chromadb google-generativeai

### Step 2: Configure Gemini API Key

In [22]:
import google.generativeai as genai
from google.colab import userdata

# Safely retrieve your Gemini API key from Google Colab Secrets
try:
    gemini_key = userdata.get('GEMINI_API_KEY')
    genai.configure(api_key=gemini_key)
    print("Gemini API key successfully configured!")
except Exception as e:
    print("Error: Please add your GEMINI_API_KEY to Colab Secrets).")

Gemini API key successfully configured!


### Step 3: PDF File Upload
Run the cell below to upload the iPhone 16 Pro / Pro Max environmental document.

In [24]:
from google.colab import files
import os

print("Please select your PDF file (e.g., the iPhone 16 Pro Environmental Report):")
uploaded = files.upload()
pdf_path = list(uploaded.keys())[0]
print(f"\nFile ready for parsing: {pdf_path}")

Please select your PDF file (e.g., the iPhone 16 Pro Environmental Report):


Saving iPhone_16_Pro_and_iPhone_16_Pro_Max_PER_Sept2024.pdf to iPhone_16_Pro_and_iPhone_16_Pro_Max_PER_Sept2024.pdf

File ready for parsing: iPhone_16_Pro_and_iPhone_16_Pro_Max_PER_Sept2024.pdf


### Step 4: Extract PDF Text
We read and extract raw textual data from the uploaded document using `pypdf`.

In [25]:
import pypdf

reader = pypdf.PdfReader(pdf_path)
pdf_text = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        pdf_text += text + "\n"

print(f"Successfully extracted {len(pdf_text)} characters from '{pdf_path}'.")

Successfully extracted 27651 characters from 'iPhone_16_Pro_and_iPhone_16_Pro_Max_PER_Sept2024.pdf'.


### Step 5: Text Chunking
We divide the parsed text into small, overlapping chunks to preserve semantic continuity.

In [26]:
def chunk_text(text, chunk_size=1000, overlap=200):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

document_chunks = chunk_text(pdf_text)
print(f"Split the document into {len(document_chunks)} discrete chunks.")

Split the document into 35 discrete chunks.


### Step 6: Initialize Local Vector Database
We set up ChromaDB to store and index our document chunks with zero network dependency.

In [29]:
### Step 6: Initialize Local Vector Database with Gemini Embeddings
import chromadb
from chromadb.api.types import Documents, Embeddings, EmbeddingFunction
import google.generativeai as genai

# Define a custom embedding function to use Google's Gemini Embedding API
class GeminiEmbeddingFunction(EmbeddingFunction):
    def __call__(self, input: Documents) -> Embeddings:
        # Call Google's native text embedding model
        response = genai.embed_content(
            model="gemini-embedding-001",
            content=input,
            task_type="retrieval_document"
        )
        return response['embedding']

# Instantiating local ephemeral vector storage client
chroma_client = chromadb.Client()
gemini_ef = GeminiEmbeddingFunction()

# Initialize or connect to the collection using Gemini Embeddings
collection = chroma_client.get_or_create_collection(
    name="Apple16",
    embedding_function=gemini_ef
)
print("Local Vector DB initialized successfully with Gemini Embeddings!")

Local Vector DB initialized successfully with Gemini Embeddings!


/tmp/ipykernel_346/3844787164.py:19: DeprecationWarning: The class GeminiEmbeddingFunction does not implement __init__. This will be required in a future version.
  gemini_ef = GeminiEmbeddingFunction()


### Step 7: Embed and Populate Database
Convert document chunks into embeddings and store them directly in our collection.

In [32]:
import time

# Map ID indexes to the text chunks
ids = [f"chunk_{i}" for i in range(len(document_chunks))]

# Safely reset/recreate collection to avoid 384 vs 3072 dimension conflicts
try:
    chroma_client.delete_collection(name="Apple16")
except Exception:
    pass

collection = chroma_client.get_or_create_collection(
    name="Apple16",
    embedding_function=gemini_ef
)

# Pacing batch insertion to stay well under Gemini's free tier rate limits (15 requests/minute)
batch_size = 5
print(f"Embedding and inserting {len(document_chunks)} chunks in throttled batches...")

for i in range(0, len(document_chunks), batch_size):
    batch_chunks = document_chunks[i:i + batch_size]
    batch_ids = ids[i:i + batch_size]

    collection.add(
        documents=batch_chunks,
        ids=batch_ids
    )
    print(f"  - Successfully uploaded chunks {i} to {min(i + batch_size, len(document_chunks))}")

    # Brief pause to prevent hitting rate limits
    if i + batch_size < len(document_chunks):
        time.sleep(5)

print("\nAll chunks successfully embedded and stored in 'Apple16' local database!")

Embedding and inserting 35 chunks in throttled batches...
  - Successfully uploaded chunks 0 to 5
  - Successfully uploaded chunks 5 to 10
  - Successfully uploaded chunks 10 to 15
  - Successfully uploaded chunks 15 to 20
  - Successfully uploaded chunks 20 to 25
  - Successfully uploaded chunks 25 to 30
  - Successfully uploaded chunks 30 to 35

All chunks successfully embedded and stored in 'Apple16' local database!


### Step 8: Context Retrieval Function
We query our local Chroma database using the query embedding to fetch the top 3 most relevant textual matches from the document.

In [36]:
def retrieve_context(query, n_results=3):
    # Retrieve document chunks relevant to the user's query
    results = collection.query(
        query_texts=[query],
        n_results=n_results
    )
    retrieved_chunks = results['documents'][0]
    # Join chunks together to build context
    return "\n---\n".join(retrieved_chunks), retrieved_chunks

### Step 9: Configure Generation Prompt
Set the instruction framework to keep the generation accurate and restricted to the retrieved context.

In [42]:
def generate_answer(query, context):
    system_prompt = (
        "You are a precise document assistant. "
        "Use only the provided document context below to answer the user's question accurately. "
        "If you do not know the answer or if it is not explicitly mentioned in the context, "
        "say that you cannot find it."
    )
    prompt = f"{system_prompt}\n\nContext:\n{context}\n\nQuestion: {query}"

    # Initialize Google Gemini for precise context generation
    model = genai.GenerativeModel("gemini-3.5-flash-lite")
    response = model.generate_content(prompt)
    return response.text

### Step 10: Run the Pipeline (Interactive Query Box)
Type your question in the input form below and run the cell to generate an answer.

In [43]:
#@markdown ### Enter Your Document Query:
user_query = "How much of the packaging is fiber-based?" #@param {type:"string"}

if not user_query.strip():
    print("Please enter a query to run the pipeline!")
else:
    print(f"Processing Query: {user_query}\n")

    # 1. Retrieve the relevant context from the local database
    context, chunks = retrieve_context(user_query)

    # 2. Display context chunks retrieved
    print("=== RETRIEVED CONTEXT SAMPLES ===")
    for idx, doc in enumerate(chunks, 1):
        print(f"Chunk {idx}: {doc[:150].strip()}...")
    print("=================================\n")

    # 3. Generate answer
    print("=== GEMINI GENERATED ANSWER ===")
    answer = generate_answer(user_query, context)
    print(answer)

Processing Query: How much of the packaging is fiber-based?

=== RETRIEVED CONTEXT SAMPLES ===
Chunk 1: work to remove 
plastic in packaging  
66% 
or more recycled content 
in fiber packaging 
100% 
of the virgin wood 
fiber in the packaging 
comes from...
Chunk 2: rbon. 
Low-carbon materials: Refers to materials created using production techniques with reduced 
carbon impact, such as Elysis (a patented technolog...
Chunk 3: Adhesives, inks, and coatings are excluded from our calculations of 
plastic content and packaging weight. 
 19  Responsible sourcing of wood fiber i...

=== GEMINI GENERATED ANSWER ===
Based on the provided context, the iPhone 16 Pro and iPhone 16 Pro Max packaging is 100 percent fiber-based.
